In [2]:
%pip install --quiet langchain_groq

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-ibm 0.1.11 requires langchain-core<0.3,>=0.2.2, but you have langchain-core 0.3.49 which is incompatible.
langchain 0.2.7 requires langchain-core<0.3.0,>=0.2.12, but you have langchain-core 0.3.49 which is incompatible.
langchain-community 0.2.1 requires langchain-core<0.3.0,>=0.2.0, but you have langchain-core 0.3.49 which is incompatible.
langchain-text-splitters 0.2.2 requires langchain-core<0.3.0,>=0.2.10, but you have langchain-core 0.3.49 which is incompatible.


In [1]:
import getpass
import os
"""
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass('Groq api key :')
from langchain.chat_models import init_chat_model

# llm = init_chat_model("deepseek-r1-distill-llama-70b", model_provider="groq", temperature=1.5)
llm = init_chat_model("llama3-70b-8192", model_provider="groq", temperature=0.7)"""

C:\Users\bocca\anaconda3\Lib\site-packages\langchain_core\_api\beta_decorator.py:87: LangChainBetaWarning: The function `init_chat_model` is in beta. It is actively being worked on, so the API may change.
  warn_beta(


ImportError: Unable to import langchain-groq. Please install with `pip install -U langchain-groq`

 # TD 3 : Chainage de LLM, Agents et applications


Au programme de ce TD :

1. Librairie Langchain
  - Découverte de la librairie et gestion des LLM
  - Présentation des outils et de workflow simples

2. Librairie Langgraph
  - Compréhension du framework de graphes LLM
  - Construction d'agents LLM simple
  - Création d'une application d'accueil de nouveaux arrivants à l'UTT.
  


---

## Introduction : Quel framework pour construire des applications basés sur des LLM ?

<u> **Langchain** </u>

Framework 'Legacy' de chainage de LLM, parmi les plus populaires.
Framework de base sur lequel repose beaucoup d'autres frameworks Agents IA

Principe de base : chaque composant d'un workflow est un objet à invoquer transformant un texte (ou un ensemble de texte) en un autre texte de façon a pouvoir chaîner les sorties des uns dans les entrée des autres.

Avantage : Très complet, permet de construire tout type de workflow.
Inconvénient : Lourd a prendre en main, difficile de se représenter des applications complexes

<u> **Langgraph** </u>

Framework construit a partir de Langchain. Les workflows sont conçus sous forme de graphes d'état. Cela permet de gérer plus facilement les workflow plus complexe, en ayant une représentation graphique.

<u> **Hors scope TD** </u>

De nombreux framework différent existe, le domaine est en constante évolution depuis 2 ans.
A titre d'exemple (non exhaustif) :
- Autogen
- Agno
- Agentscope
- Agentstudio
- Haystack



---

##Partie 1 : Chaînage de LLM avec langchain


### Entrée/Sortie du llm

<u> **Le régime alimentaire du LLM** </u>

- Avant 2022 (jusque GPT3) : principalement string $\rightarrow$ string. Le LLM cherche à continuer un texte.
- 2022 (génération GPT3.5) : Dictionnaire de messages (user message, assistant message). Le LLM s'inspire des conversations humaines pour générer des messages.
- 2023 (génération GPT4) : Messages plus structurés, avec plus de rôles. Formats JSON plus complexes pris en compte.
- Fin 2023 : modèles multi-modaux.
- 2024 (génération 4o et +) : Ajout d'input spécifiques pour des outils, Chaîne de pensées, etc ...

Avec une augmentation continue des tailles de contexte gérée par les LLM.


<u> **Formatting des inputs avec des PromptTemplate / Messages** </u>





In [ ]:
from langchain_core.prompts import ChatPromptTemplate
prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant of a video game store.",
        ),
        ("human", "Explain me  how to win this game : {game}"),
    ]
)

prompt_template.invoke('Zelda')

In [ ]:
llm_with_prompt = prompt_template | llm

In [ ]:
llm.invoke('Zelda')

In [ ]:
llm_with_prompt.invoke('Zelda')

Les principaux types de messages :
- SystemMessage : Permet de cadrer le LLM. "Définir sa personnalité"
- HumanMessage : C'est vous, principalement du texte brut comme ce que vous savez faire
- AiMessage : La réponse du LLM. Peut être du texte, ou bien des sorties formatées

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

In [ ]:
messages = [SystemMessage('Tu es un dirigeant de Thales qui doit répondre devant les actionnaires des actions du groupe. Tu parles a un employé du groupe.'),
            HumanMessage('Quel augmentation meriterais-je, sans explications ?')]
llm.invoke(messages).pretty_print()

messages = [SystemMessage('Tu es un syndicaliste de Thales qui veut augmenter les droits des employés. Tu parles a un employé du groupe.'),
            HumanMessage('Quel augmentation meriterais-je, sans explications ?')]
llm.invoke(messages).pretty_print()


<u> **Formatting des outputs : Sortie dans des formats spécifiables** </u>


On va créer des objets permettant de réunir l'ensemble des données recherchées.
On recherchera l'équilibre entre la précision des noms et demandes (pour la qualité de la réponse générée), et l'excès de gourmandise (le llm n'est pas un super héros, il peut être (souvent même) flemmard si on lui en demande trop)


In [ ]:
from pydantic import BaseModel, Field
from typing import List
from langchain_core.messages import HumanMessage, SystemMessage


# Schema for structured output to use in planning
class Jeux(BaseModel):
    name: str = Field(description="Name of game")
    overview: str = Field(description="Brief overview of the game.")
    how_to_win: str = Field(description="How to win the game.")
    rate: str = Field(description="Rate this game")


class BibliothequeJeux(BaseModel):
    games: List[Jeux] = Field(description="List of video games.")


# Augment the LLM with schema for structured output
help_games = llm.with_structured_output(BibliothequeJeux)

In [ ]:
help_games.invoke("all age of empire games")

Exercices :

1. Faire un formating d'output qui me permette de lister les théoremes célèbres en théorie des groupes avec leurs énoncés, date et mathématicien

2. Faire une fonction qui prends en entrée un pays et qui retourne en sortie un dictionnaire d'informations géographiques du pays.

In [ ]:
class Theoreme(BaseModel):
    enonce: str = Field(description="énoncé du théorème")
    date: str = Field(description="date du théorème")
    mathematicien: str = Field(description="nom du mathematicien")
    
class BibliothequeTheoreme(BaseModel):
    games: List[Theoreme] = Field(description="List of theorems.")

math_explainer = llm.with_structured_output(BibliothequeTheoreme)

In [ ]:
math_explainer.invoke('List all mathematics theorem in group theory')

In [ ]:
class Pays(BaseModel):
    name: str = Field(description="name of the country")
    superficie: str = Field(description="superficie of the country")

info_pays = llm.with_structured_output(Pays)

def get_geographic_information(country):
    question = "Give me information about this country : " + country
    return info_pays.invoke(question)

In [ ]:
for country in ['France', 'Thailand', 'Ouganda']:
    print(get_geographic_information(country))

 ### Amélioration du LLM avec des Outils

Comme nous l'avons vu, un LLM va générer un texte, éventuellement formaté et parsé. En revanche, on reste sur un outil probabiliste qui génère (comme vous le feriez) un texte parmi ceux les plus probables.

- Combien font $6 \times 7$ : Vous me répondrez presque tous 42.
- Combien font $12563 \times 701452$ : Vous me répondrez "dans les $87\ 000\ 000\  000$"


In [ ]:
print(6 * 7, 12563 * 7014527)
llm.invoke('Compute 6 * 7').pretty_print()
llm.invoke('Compute 12563 * 7014527').pretty_print()

En vrai ? Vous auriez pris votre calculatrice sur votre portable !
Donnons cette opportunité au LLM !

In [ ]:
query = "What is 12563 * 7014527? Also, what is 15 + 56?"

12563 * 7014527, 15 + 56

In [ ]:
from langchain_core.messages import HumanMessage

messages = [HumanMessage(query)]
llm.invoke(messages).pretty_print()


Introduction de fonction python vues comme des outils par le LLM :

In [ ]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Adds a and b."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiplies a and b."""
    return a * b

tools = [add, multiply]

llm_with_tools = llm.bind_tools(tools)

In [ ]:
messages = [HumanMessage(query)]
ai_message = llm_with_tools.invoke(messages)
ai_message.pretty_print()
messages.append(ai_message)



**Le LLM ne donne pas de réponse mais des demandes d'appels aux fonctions qu'on lui a donné**

In [ ]:
for tool_call in ai_message.tool_calls:
    selected_tool = {"add": add, "multiply": multiply}[tool_call["name"].lower()]
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
messages

**Les outils ont fait leur boulot, on redonne la réponse au llm pour générer un texte humain**

In [ ]:
llm_with_tools.invoke(messages).pretty_print()

<u> **Exemple d'outils : Execution de code python.** </u>

Nous passons par une API qui execute du code python dans une sandbox, pour ne pas donner les pleins pouvoirs au llm sur son ordi.
- Aller se creer un compte sur https://dashboard.riza.io/ et récupérer une clé API

In [ ]:
# Installation de rizaio
%pip install --upgrade --quiet langchain-community rizaio

In [ ]:
if not os.environ.get("RIZA_API_KEY"):
    os.environ["RIZA_API_KEY"] = getpass.getpass('Riza api key :')


In [ ]:
from langchain_community.tools.riza.command import ExecPython

In [ ]:
tools = [ExecPython()]

In [ ]:

prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Make sure to use a tool if you need to solve a problem.",
        ),
        ("human", "{input}"),
    ]
)

llm_with_tools = llm.bind_tools(tools)
llm_with_tools_with_prompt = prompt_template | llm_with_tools

In [ ]:
result = llm_with_tools.invoke("how many rs are in strawberry?")

for tool_call in result.tool_calls:
    selected_tool = {"riza_exec_python": tools[0]}[tool_call["name"].lower()]
    print(tool_call)
    tool_msg = selected_tool.invoke(tool_call)
    tool_msg.pretty_print()

**Automatisation de l'appel a l'outil et de la génération de réponse dans des fonctions du framework langchain**

On va utiliser les fonctions existantes du framework pour automatiser ces étapes, et être capable de relancer la réponse si l'outil ne donne pas satisfaction

In [ ]:
from langchain.agents import AgentExecutor, create_tool_calling_agent


prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Make sure to use a tool if you need to solve a problem.",
        ),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

agent = create_tool_calling_agent(llm, tools, prompt_template)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
agent_executor.invoke({'input': "how many rs are in strawberry?"})

<u> **Exercices** </u> :
(Liste des outils langchain dispo : https://python.langchain.com/docs/integrations/tools/)

1. Calculer le 51 ème nombre de fibonnaci
2. Estimer le nombre d'habitant en 2024 dans une ville à partir de wikipedia



In [ ]:
prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Make sure to use a tool if you need to solve a problem.",
        ),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

agent = create_tool_calling_agent(llm, tools, prompt_template)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

agent_executor.invoke({'input': "Give me the 51th number of Fibonacci."})

In [ ]:
get_number_inhabitants('Paris')
get_number_inhabitants('Roscoff')
get_number_inhabitants('Falaise')

---

## Partie 2: Application LLM-based avec langgraph

Chainage des LLM sous forme de graphe d'état.

Pour résumer, l'utilisation d'un graphe permet deux choses principales :
- Réaliser le chainage des LLM avec un apercu graphique possible.
- Gérer la mémoire avec des variables d'état du graphe qui sont modifiées par les LLM

L'utilisation de langgraph (ou de tout autre framework équivalent) permet alors de construire de façon plus robuste des applications faisant intervenir des workflows de plus en plus complexe

**Exemple de graphe d'appel a un LLM simple**

Le principe : Chaque noeud du graphe est un appel à une fonction python. Cette fonction opère sur l'état du graphe et fait appel à des LLM ou d'autres outils

In [ ]:
%pip install --upgrade --quiet langgraph

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, MessagesState, START, END


def call_model(state: MessagesState):
    print(state["messages"])
    response = llm.invoke(state["messages"])
    return {"messages": response}


builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")
builder.add_edge("call_model", END)
graph = builder.compile()

graph

In [ ]:
input_message = {"role": "user", "content": "hi! I'm bob"}
for chunk in graph.stream({"messages": [input_message]}, stream_mode="values"):
    chunk["messages"][-1].pretty_print()


**Graph exemple d'appel d'outil**

In [ ]:
from langchain_core.messages import AIMessage
from langchain_core.tools import tool

from langgraph.prebuilt import ToolNode

In [ ]:
tools = [add, multiply]
tool_node = ToolNode(tools)

model_with_tools = llm.bind_tools(tools)

In [ ]:
from typing import Literal
from langchain_core.messages import AnyMessage
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, MessagesState, START, END
from typing_extensions import Annotated



def should_continue(state: MessagesState):
    messages = state["messages"]
    last_message = messages[-1]
    if last_message.tool_calls:
        return "tools"
    return END


def call_model(state: MessagesState):
    messages = state["messages"]
    response = model_with_tools.invoke(messages)
    return {"messages": response}


workflow = StateGraph(MessagesState)

# Define the two nodes we will cycle between
workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue, ["tools", END])
workflow.add_edge("tools", "agent")

app = workflow.compile()
app

In [ ]:
# example with a single tool call
for chunk in app.stream({"messages": [("human", query)]},stream_mode='values'):
  print(chunk)
  chunk["messages"][-1].pretty_print()


### Exercice :

Créer une application du style "akinator"

In [ ]:
from langchain.agents import load_tools
tools = load_tools(["human"])
human_talk_llm = llm.bind_tools(tools)
human_node = ToolNode(tools)

In [ ]:
tools[0].run('Hello ?')

In [ ]:
from typing import Literal
from langchain_core.messages import AnyMessage
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, MessagesState, START, END
from typing_extensions import Annotated



def should_continue(state: MessagesState):
    messages = state["messages"]
    last_message = messages[-1]
    if last_message.tool_calls:
        return "tools"
    return END


def call_model(state: MessagesState):
    messages = state["messages"]
    response = model_with_tools.invoke(messages)
    return {"messages": response}


workflow = StateGraph(MessagesState)

# Define the two nodes we will cycle between
workflow.add_node("agent", call_model)
workflow.add_node("tools", human_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue, ["tools", END])
workflow.add_edge("tools", "agent")

app = workflow.compile()
app

In [ ]:
for chunk in app.stream({"message" : [("system", "You are a genius guesser. You can ask me some question, to find which knonw personnality I am, I can only answer yes or no. Stop only if you have the right person.")]}):
    print(chunk)
    chunk["message"][-1].pretty_print()

### Application :

1. Créer une application allant chercher sur internet des informations sur le site de l'utt

2. Faire un chatbot associé d'accueil pour la nouvelle promotion.

In [ ]:
!pip install --quiet langchain_tavily
import getpass
import os
if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Tavily API key:\n")

In [ ]:
from langchain_tavily import TavilySearch

tool = TavilySearch(
    max_results=5,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

In [ ]:
tool.invoke("Quel age a google ? Include only www.google.fr sources")